# **2일차 팀 프로젝트: 문서 기반 RAG 시스템 구축**

## 프로젝트 목표
1. 팀에서 선정한 PDF 문서를 Qdrant Cloud에 저장
2. Parent Document Retriever 패턴 적용
3. 검색 테스트 및 RAG 시스템 구현

## 구현 단계
- 환경 설정 확인
- PDF 문서 로딩
- Child Chunk 생성 및 Qdrant Cloud 저장
- Parent Document 저장
- 검색 테스트
- RAG 시스템 구현 및 테스트

## 0. 환경 변수 설정

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Qdrant Cloud 설정 확인
if os.environ.get("QDRANT_URL") and os.environ.get("QDRANT_API_KEY"):
    print("✓ Qdrant Cloud 설정이 완료되었습니다.")
    print(f"  URL: {os.environ.get('QDRANT_URL')}")
else:
    print("✗ Qdrant Cloud 설정이 필요합니다.")
    print("  .env 파일에 QDRANT_URL과 QDRANT_API_KEY를 추가하세요.")

✓ OpenAI API Key가 설정되었습니다.
✓ Qdrant Cloud 설정이 완료되었습니다.
  URL: https://01517885-5c59-4d08-8ed4-17006164d017.us-east-1-1.aws.cloud.qdrant.io


## 1. PDF 문서 로딩

**TODO: 팀에서 선정한 PDF 파일 경로를 입력하세요**

In [1]:
from langchain_core.documents import Document
import fitz

# TODO: PDF 파일 경로를 입력하세요
# 예시: "../datasets/your_document.pdf"
file_path = "YOUR_PDF_FILE_PATH_HERE"

doc = fitz.open(r"C:\Users\khm35\Downloads\[고용노동부] 2026년 노무관리 가이드북__.pdf")
docs = []

# 페이지 단위로 Document 생성 (Parent Document)
for page_num in range(len(doc)):
    page = doc[page_num]
    text = page.get_text("text", sort=True)

    # 빈 페이지는 스킵
    if len(text.strip()) < 10:
        continue

    docs.append(
        Document(
            page_content=text,
            metadata={
                "source": file_path.split("/")[-1],
                "page": page_num + 1,
                "parent_id": f"page_{page_num + 1}"
            }
        )
    )

doc.close()

print(f"총 {len(docs)}개의 페이지(Parent Document) 로드 완료")
print(f"\n첫 번째 페이지 길이: {len(docs[0].page_content)}자")
print(f"평균 페이지 길이: {sum(len(d.page_content) for d in docs) / len(docs):.0f}자")

# 첫 페이지 내용 미리보기
print(f"\n첫 페이지 내용 미리보기:")
print(docs[0].page_content[:300] + "...")

총 264개의 페이지(Parent Document) 로드 완료

첫 번째 페이지 길이: 185자
평균 페이지 길이: 1159자

첫 페이지 내용 미리보기:
www.moel.go.kr


                        2026
           핵심만 담은
     노무관리
     가이드 북
                            노무관리지도 자가진단표와 함께 보실 수
                             있도록 쉽게 쓴 실용 가이드 북입니다....


## 2. Child Chunk 생성

**TODO: 청킹 전략을 조정해보세요 (선택사항)**
- chunk_size: 각 청크의 크기 (기본 400자)
- chunk_overlap: 청크 간 겹치는 부분 (기본 50자)

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TODO: 필요시 chunk_size와 chunk_overlap 값을 조정하세요
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,      # 작은 크기로 정확한 검색
    chunk_overlap=50     # 문맥 유지
)

# Parent를 Child chunk로 분할
child_docs = []

for parent_doc in docs:
    chunks = child_splitter.split_text(parent_doc.page_content)

    for chunk in chunks:
        child_docs.append(
            Document(
                page_content=chunk,
                metadata={
                    "parent_id": parent_doc.metadata["parent_id"],
                    "page": parent_doc.metadata["page"],
                    "source": parent_doc.metadata["source"]
                }
            )
        )

print(f"\n생성된 통계:")
print(f"  - Parent 문서 수: {len(docs)}")
print(f"  - Child chunk 수: {len(child_docs)}")
print(f"  - 평균 chunk/page: {len(child_docs) / len(docs):.1f}")

# Child chunk 샘플 확인
print(f"\nChild chunk 샘플 (첫 3개):")
for i in range(min(3, len(child_docs))):
    print(f"\nChunk {i + 1}:")
    print(f"  Parent ID: {child_docs[i].metadata['parent_id']}")
    print(f"  Page: {child_docs[i].metadata['page']}")
    print(f"  Length: {len(child_docs[i].page_content)}자")
    print(f"  Content: {child_docs[i].page_content[:100]}...")


생성된 통계:
  - Parent 문서 수: 264
  - Child chunk 수: 1128
  - 평균 chunk/page: 4.3

Child chunk 샘플 (첫 3개):

Chunk 1:
  Parent ID: page_1
  Page: 1
  Length: 185자
  Content: www.moel.go.kr


                        2026
           핵심만 담은
     노무관리
     가이드 북
               ...

Chunk 2:
  Parent ID: page_2
  Page: 2
  Length: 380자
  Content: 2026
핵심만
 담은
노무관리
가이드 북

              1 근로조건 서면명시

                             1-1 근로조건 서면명시      ...

Chunk 3:
  Parent ID: page_2
  Page: 2
  Length: 313자
  Content: 3 임금 등 각종 금품 지급

                             3-1 금품청산                                  032

       ...


## 3. Qdrant Cloud에 Child Chunk 저장

**TODO: 컬렉션 이름을 팀 프로젝트에 맞게 변경하세요**

In [5]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from uuid import uuid4

# Qdrant Cloud 클라이언트 생성
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)

print("Qdrant Cloud에 연결되었습니다.")
print(f"  URL: {os.getenv('QDRANT_URL')}")

Qdrant Cloud에 연결되었습니다.
  URL: https://01517885-5c59-4d08-8ed4-17006164d017.us-east-1-1.aws.cloud.qdrant.io


In [6]:
# 임베딩 함수
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# TODO: 팀 프로젝트에 맞는 컬렉션 이름으로 변경하세요
# 예시: "team1_healthcare_docs", "team2_legal_docs" 등
collection_name = "노동자_근로기준_상담"

# 컬렉션 존재 여부 확인
collections = client.get_collections().collections
existing_collection = any(col.name == collection_name for col in collections)

if existing_collection:
    print(f"컬렉션 '{collection_name}'이 이미 존재합니다.")
    user_input = input("기존 데이터를 삭제하고 새로 추가하시겠습니까? (y/n): ")

    if user_input.lower() == 'y':
        print(f"컬렉션 '{collection_name}' 삭제 중...")
        client.delete_collection(collection_name=collection_name)
        print("컬렉션이 삭제되었습니다.")

        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
        )
        print(f"컬렉션 '{collection_name}' 생성 완료")
    else:
        print("기존 컬렉션을 사용합니다.")
else:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
    )
    print(f"컬렉션 '{collection_name}' 생성 완료")

# 벡터스토어 생성
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings
)

# Child chunk 추가
uuids = [str(uuid4()) for _ in range(len(child_docs))]
vectorstore.add_documents(documents=child_docs, ids=uuids)

print(f"\n{len(child_docs)}개의 Child chunk가 Qdrant Cloud에 추가되었습니다.")

컬렉션 '노동자_근로기준_상담'이 이미 존재합니다.
컬렉션 '노동자_근로기준_상담' 삭제 중...
컬렉션이 삭제되었습니다.
컬렉션 '노동자_근로기준_상담' 생성 완료

1128개의 Child chunk가 Qdrant Cloud에 추가되었습니다.


## 4. Parent Document 저장 (Docstore)

In [7]:
# Parent 문서를 dict에 저장 (parent_id를 키로 사용)
parent_docstore = {}

for parent_doc in docs:
    parent_id = parent_doc.metadata["parent_id"]
    parent_docstore[parent_id] = parent_doc

print(f"Docstore에 {len(parent_docstore)}개의 Parent 문서 저장 완료")
print(f"\nDocstore 키 예시: {list(parent_docstore.keys())[:5]}")

Docstore에 264개의 Parent 문서 저장 완료

Docstore 키 예시: ['page_1', 'page_2', 'page_3', 'page_4', 'page_5']


## 5. Parent Document Retriever 구현

In [8]:
from typing import List

class ParentDocumentRetriever:
    """
    Parent Document Retriever 직접 구현

    원리:
    1. vectorstore에서 child chunk 검색
    2. child chunk의 parent_id 추출
    3. docstore에서 parent_id로 parent 문서 반환
    """

    def __init__(self, vectorstore, parent_docstore, k: int = 2):
        self.vectorstore = vectorstore
        self.parent_docstore = parent_docstore
        self.k = k

    def invoke(self, query: str) -> List[Document]:
        # 1. Vectorstore에서 child chunk 검색
        child_results = self.vectorstore.similarity_search(query, k=self.k)

        # 2. Child chunk에서 parent_id 추출 (중복 제거)
        parent_ids = []
        for doc in child_results:
            parent_id = doc.metadata.get("parent_id")
            if parent_id and parent_id not in parent_ids:
                parent_ids.append(parent_id)
                if len(parent_ids) >= self.k:
                    break

        # 3. Docstore에서 parent 문서 가져오기
        parent_docs = []
        for parent_id in parent_ids:
            if parent_id in self.parent_docstore:
                parent_docs.append(self.parent_docstore[parent_id])

        return parent_docs

    def get_child_chunks(self, query: str, k: int = 3) -> List[Document]:
        """비교용: Child chunk 직접 반환"""
        return self.vectorstore.similarity_search(query, k=k)

# Retriever 생성
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    parent_docstore=parent_docstore,
    k=2
)

print("✓ Parent Document Retriever 생성 완료")

✓ Parent Document Retriever 생성 완료


## 6. 검색 테스트

**TODO: 팀 문서에 맞는 질문으로 변경하여 검색 테스트를 진행하세요**

In [9]:
# TODO: 팀 문서에 맞는 검색 질문을 작성하세요
query = "근로기준법상 근로시간과 휴게시간에 대한 규정은 무엇인가요?"

print(f"검색 쿼리: {query}\n")
print("="*80)

# Child chunk 검색
print("\n[1] Child Chunk 검색 결과")
print("-"*80)
child_results = parent_retriever.get_child_chunks(query, k=2)

for i, result in enumerate(child_results, start=1):
    print(f"\nChunk {i}:")
    print(f"  페이지: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용: {result.page_content}")

# Parent document 검색
print("\n" + "="*80)
print("\n[2] Parent Document 검색 결과")
print("-"*80)
parent_results = parent_retriever.invoke(query)

for i, result in enumerate(parent_results, start=1):
    print(f"\nPage {i}:")
    print(f"  페이지 번호: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용 미리보기: {result.page_content[:300]}...")

검색 쿼리: 근로기준법상 근로시간과 휴게시간에 대한 규정은 무엇인가요?


[1] Child Chunk 검색 결과
--------------------------------------------------------------------------------

Chunk 1:
  페이지: 67
  Parent ID: page_67
  길이: 323자
  내용: 법에서 정한 기준근로시간

                                 - 통상근로자는 1일 8시간, 1주 40시간임

                                 - 다만, 법에서는 근로자의 연령 또는 작업 성질에 따라 법정

                 근로시간을 달리 정함

         법정근로시간        구분            의미

                      통상근로자        1일 8시간, 1주 40시간

                       연소근로자(18세 미만)     1일 7시간, 1주 35시간

Chunk 2:
  페이지: 66
  Parent ID: page_66
  길이: 334자
  내용: 3    법정근로시간을 준수하고 있습니까?              67쪽




           근로기준법법 규정

             제2조(정의)

          ① 이 법에서 사용하는 용어의 뜻은 다음과 같다.

            8.“소정(所定)근로시간”이란 제50조, 제69조 본문 또는 「산업안전보건법」

            제139조 제1항에 따른 근로시간의 범위에서 근로자와 사용자 사이에 정한

          근로시간을 말한다.

            제50조(근로시간)

          ① 1주간의 근로시간은 휴게시간을 제외하고 40시간을 초과할 수 없다.


[2] Parent Document 검색 결과
----------------------------------------------------

## 7. RAG 시스템 구현

**TODO: 시스템 프롬프트를 팀 문서에 맞게 수정하세요**

In [10]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from IPython.display import Markdown, display
from pydantic import BaseModel, Field
from typing import Optional

llm = init_chat_model("gpt-5.4-mini")

# 카테고리 분류 결과를 위한 Pydantic 모델
class CategoryClassification(BaseModel):
    """천안시 정책 카테고리 분류 결과"""
    category: Optional[str] = Field(
        description="선택된 카테고리 이름. 적합한 카테고리가 없으면 None"
    )

def determine_category(question: str) -> Optional[str]:
    """
    LLM을 사용하여 질문을 분석하고 적절한 노동자 근로기준법 관련 카테고리를 결정합니다.

    Args:
        question: 사용자 질문

    Returns:
        카테고리 이름 (문자열) 또는 None (필터 없음)
    """

    # 사용 가능한 카테고리 목록
    available_categories = {
        "근로계약_서류관리": "근로계약서 작성 및 교부, 근로조건 서면명시, 기간제·단시간 근로자 계약, 근로자명부, 계약서류 보존, 임금대장 관련",
        "임금_수당_금품": "임금 지급, 임금명세서, 금품청산, 휴업수당, 연장·야간·휴일근로 수당, 임금체불 및 각종 금품 지급 관련",
        "근로시간_휴게": "법정근로시간, 소정근로시간, 연장근로 한도, 초과근로, 휴게시간, 대기시간 관련",
        "휴일_연차휴가": "주휴일, 주휴수당, 공휴일, 대체공휴일, 유급휴일, 연차유급휴가, 연차 미사용수당 관련",
        "임신_출산_육아_모성보호": "임산부 보호, 여성·연소자 근로, 출산휴가, 배우자 출산휴가, 육아휴직, 육아기 근로시간 단축 관련",
        "취업규칙_사업장규정": "취업규칙 작성·신고·변경, 법령 및 단체협약 준수, 사업장 내부 근로조건과 규정 관련",
        "퇴직_퇴직급여": "퇴직금, 확정급여형·확정기여형 퇴직연금, 중소기업퇴직연금기금제도, 퇴직급여 지급 관련",
        "직장내_권리보호": "직장 내 괴롭힘, 직장 내 성희롱, 고용상 성차별, 비정규직 차별 및 근로자 권리 보호 관련",
        "최저임금": "최저임금 적용, 최저임금 계산, 수습·인턴 최저임금, 최저임금 산입범위, 최저임금 주지의무 관련",
        "노사관계_노사협의회": "노사협의회 설치와 운영, 노사협의회 회의, 고충처리, 근로자위원·사용자위원 및 노사 협의 관련",
    }

    # LLM에게 카테고리 분류 요청
    category_list = "\n".join([f"- {cat}: {desc}" for cat, desc in available_categories.items()])

    classification_prompt = f"""다음 질문을 분석하여 가장 적합한 천안시 정책 카테고리를 선택하세요.

<available_categories>
{category_list}
</available_categories>

<question>
{question}
</question>

<rules>
1. 질문의 주요 주제와 가장 관련 있는 카테고리를 선택하세요
2. 여러 카테고리가 관련될 수 있지만, 가장 핵심적인 하나만 선택하세요
3. 적합한 카테고리가 없거나 매우 일반적인 질문이면 category를 null로 설정하세요
</rules>
"""

    # Structured Output을 사용하여 LLM 호출
    structured_llm = llm.with_structured_output(CategoryClassification)
    result = structured_llm.invoke(classification_prompt)

    print(f"[LLM 분류 결과]")
    print(f"  카테고리: {result.category}")

    return result.category


def rag_with_dynamic_filter(question: str) -> str:
    """
    동적 필터링을 적용한 RAG
    """
    # 1. 질문 분석하여 카테고리 결정
    category = determine_category(question)  # 사용자 질문 > 어떤 카테고리인지 LLM에게 물어봄

    # 2. 필터 설정
    search_kwargs = {"k": 3}
    if category:
        search_kwargs["filter"] = models.Filter(
            must=[
                models.FieldCondition(
                    key="metadata.category",
                    match=models.MatchValue(value=category)
                )
            ]
        )
        print(f"✓ 적용된 필터: category = '{category}'\n")
    else:
        print(f"✓ 필터 없음 (전체 문서 검색)\n")

    # 3. 문서 검색
    retriever = vectorstore.as_retriever(search_kwargs=search_kwargs)  # retriever > invoke
    retrieved_docs = retriever.invoke(question)

    # 4. 컨텍스트 구성
    context_parts = []
    for doc in retrieved_docs:
        page = doc.metadata['page']
        cat = doc.metadata['category']
        context_parts.append(
            f"[출처: {doc.metadata['source']}, 페이지: {page}, 카테고리: {cat}]\n{doc.page_content}"
        )

    context = "\n\n---\n\n".join(context_parts)

    # 5. 프롬프트 생성
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    response = llm.invoke(formatted_prompt)
    return response.content


# TODO: 시스템 프롬프트를 팀 문서 도메인에 맞게 수정하세요
# 예시: "당신은 의료 전문가입니다.", "당신은 법률 전문가입니다." 등
template = """
당신은 아르바이트생의 노동 권리를 상담해주는 근로기준법 전문가입니다.
상담자는 법을 잘 모르는 아르바이트생입니다. 어려운 법률 용어는 반드시 쉬운 말로 풀어서 설명하세요.

[답변 규칙]
1. 반드시 아래 [참고 정보]에 있는 내용만 근거로 답하세요.
   참고 정보에 없는 내용은 지어내지 말고 "제가 가진 자료에는 없는 내용입니다"라고 솔직히 말하세요.
2. 참고 정보는 사업주(사장님)를 대상으로 쓰인 문서입니다.
   따라서 "사용자는 ~해야 한다"는 내용은 "사장님은 ~할 의무가 있으니, 요구할 수 있습니다"처럼
   알바생 입장으로 바꿔서 설명하세요.
3. 답변 끝에 참고한 문서의 출처와 페이지 번호를 반드시 표시하세요.
4. 금액, 시간, 인원수 같은 숫자는 참고 정보에 적힌 그대로만 쓰고, 직접 계산하지 마세요.
5. 상황에 따라 달라질 수 있는 문제라면 그 점을 알려주고,
   고용노동부 상담센터(국번없이 1350)에 문의하도록 안내하세요.

[답변 형식]
- 결론부터 한두 문장으로 먼저 말하기
- 그 다음 이유를 쉬운 말로 설명하기
- 마지막에 출처 표시

[참고 정보]
{context}

[질문]
{question}

[답변]
"""

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)


def rag_with_parent_retriever(question: str) -> str:
    """
    Parent Document Retriever를 사용한 RAG
    """
    # 1. 문서 검색
    retrieved_docs = parent_retriever.invoke(question)

    # 2. 컨텍스트 구성
    context_parts = []
    for doc in retrieved_docs:
        source = doc.metadata.get("source", "?")
        page_num = doc.metadata.get("page", "?")
        context_parts.append(f"[출처: {source}, 페이지: {page_num}]\n{doc.page_content}")

    context = "\n\n---\n\n".join(context_parts)

    # 3. 프롬프트 생성
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    # 4. LLM 호출
    response = llm.invoke(formatted_prompt)
    return response.content


print("✓ RAG 시스템 준비 완료")

✓ RAG 시스템 준비 완료


## 8. RAG 시스템 테스트

**TODO: 팀 문서에 맞는 다양한 질문으로 RAG 시스템을 테스트하세요**

In [11]:
# TODO: 팀 문서에 맞는 질문들을 작성하세요
questions = [
    "알바생 주휴수당 기준 알려줘",
    "알바생 근로시간과 휴게시간 규정에 대해 알려줘",
    "알바생 최저임금과 관련된 법규에 대해 알려줘"
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print(f"{'='*80}\n")

    answer = rag_with_parent_retriever(q)
    display(Markdown(answer))


질문: 알바생 주휴수당 기준 알려줘



결론부터 말하면, **소정근로일을 개근하면 1일을 유급휴일로 받을 수 있고, 지각·조퇴·외출은 결근이 아닙니다.** 다만 **휴직 중에는 유급휴일을 청구할 수 없으니**, 본인 상황이 주휴수당 대상인지가 애매하면 **고용노동부 상담센터 1350**에 문의하는 게 좋습니다.

쉬운 말로 설명하면, 참고 자료에는 **사장님은 알바생이 정해진 근무일을 빠지지 않고 일했을 때 1일을 유급휴일로 주어야 한다**고 되어 있습니다. 즉, **주중에 하루를 빼먹은 경우에는 무급휴일**로 볼 수 있고, **지각·조퇴·외출은 결근으로 보지 않으므로 유급휴일 판단에서 결근과는 다르게 취급됩니다.**  
또한 참고 자료에는 **휴직은 일을 제공하지 않는 상태이므로 유급휴일 청구권이 없다**고 되어 있습니다.

주휴수당 금액은 참고 자료에 따라 이렇게 설명할 수 있습니다.  
- **주휴수당 = 1일 소정근로시간 수 × 시간급 임금**  
- 다만 **사업장의 근로시간이 법정근로시간을 초과하는 경우에는 법정근로시간에 해당하는 임금만 주휴수당으로 지급**합니다.  
- 예시로, **1주에 5일, 각 6시간** 일하고 **시간급 11,000원**이면 **주휴수당은 66,000원**입니다.  
- 참고 자료에는 **1일 9시간, 1주 45시간**을 일하더라도 주휴수당은 **법정근로시간인 8시간 분만 지급**한다고 되어 있습니다.

정리하면, 알바생 입장에서는 **“소정근로일을 다 채웠는지”가 핵심**이고, 그 조건을 충족하면 **사장님에게 1일 유급휴일 지급을 요구할 수 있습니다.**  
다만 **내 근무 형태가 단시간 근로자인지, 휴직인지, 또는 법정근로시간을 넘는지에 따라 달라질 수 있으니**, 정확한 확인은 **고용노동부 상담센터 1350**에 문의하세요.

출처: **[출처: YOUR_PDF_FILE_PATH_HERE, 페이지: 99], [출처: YOUR_PDF_FILE_PATH_HERE, 페이지: 52]**


질문: 알바생 근로시간과 휴게시간 규정에 대해 알려줘



결론부터 말씀드리면, 알바생의 **근로시간과 휴게시간은 사장님이 취업규칙에 정해 두어야 하는 내용**이라서, 내가 일하는 시간과 쉬는 시간이 정리되어 있는지 확인할 수 있습니다. 특히 **단시간근로자**라면 **근로일과 근로일별 근로시간**이 반드시 적혀 있어야 합니다.

이유를 쉽게 설명하면, 참고자료에는 **상시 10명 이상의 근로자를 쓰는 사장님은 취업규칙을 작성해서 고용노동부에 신고해야 하고**, 그 취업규칙에는 **업무의 시작과 종료 시각, 휴게시간**이 들어가야 한다고 되어 있습니다. 즉, 알바생 입장에서는 “사장님은 근무 시작 시간, 끝나는 시간, 쉬는 시간을 정해 둘 의무가 있으니 그 내용이 취업규칙에 있는지 요구할 수 있다”는 뜻입니다.

또한 자료에는 **단시간근로자의 경우 “근로일 및 근로일별 근로시간”을 반드시 기재해야 한다**고 되어 있습니다. 예시로는  
- 주 5일, 일 6시간인 경우: 업무 시작 시각 09시 00분, 업무 종료 시각 16시 00분, 휴게 시간 12시 00분부터 13시 00분까지  
- 주 2일, 일 4시간인 경우: 업무 시작 시각 20시 00분, 업무 종료 시각 다음 날 0시 30분, 휴게 시간 22시 00분부터 22시 30분까지  
처럼 적는 방식이 나와 있습니다.  
다만, 내 상황이 **정확히 어떤 형태의 단시간근로인지**, 그리고 **취업규칙에 실제로 어떻게 적혀 있어야 하는지**는 근무 형태에 따라 달라질 수 있습니다. 그런 부분은 고용노동부 상담센터 **1350**에 문의하시면 됩니다.

출처: **2026 핵심만 담은 노무관리 가이드 북, 146쪽 / 20쪽**


질문: 알바생 최저임금과 관련된 법규에 대해 알려줘



결론부터 말하면, **사장님은 2026년 적용 최저임금 게시물을 직장에 게시해야 할 의무가 있으니, 알바생은 이 게시가 되어 있는지 요구할 수 있습니다.**  
다만 **최저임금의 구체적인 금액이나 자세한 법규 내용은 제가 가진 자료에는 없는 내용입니다.**

이유를 쉽게 말하면, 참고 자료에는 **“최저임금 주지의무”**와 **“2026년 적용 최저임금 게시물”**이 적혀 있어서, 사장님이 최저임금 관련 내용을 **알바생이 볼 수 있게 알려줘야 한다는 점**만 확인할 수 있습니다.  
하지만 이 자료만으로는 **최저임금이 얼마인지, 어떤 경우에 적용되는지, 위반하면 어떻게 되는지**는 알 수 없습니다. 이런 부분은 상황에 따라 달라질 수 있으니, 정확한 확인이 필요하면 **고용노동부 상담센터 1350**에 문의하시는 게 좋습니다.

**출처:** YOUR_PDF_FILE_PATH_HERE, p.207 / p.209

## 프로젝트 점검 체크리스트

**완료한 항목을 확인하세요:**

- [ ] PDF 문서 선정 및 로딩 완료
- [ ] Child Chunk 생성 완료
- [ ] Qdrant Cloud에 데이터 저장 완료
- [ ] Parent Document Retriever 구현 완료
- [ ] 검색 테스트 완료 (Child vs Parent 비교)
- [ ] RAG 시스템 구현 완료
- [ ] 최소 3개 이상의 질문으로 테스트 완료
- [ ] 시스템 프롬프트 도메인에 맞게 수정 완료

---

## 추가 개선 아이디어

1. **청킹 전략 최적화**: chunk_size와 chunk_overlap 조정
2. **검색 개수 조정**: retriever의 k 값 변경
3. **프롬프트 개선**: 더 구체적인 답변 형식 지정
4. **메타데이터 활용**: 날짜, 카테고리 등 추가 필터링
5. **하이브리드 검색**: 키워드 + 벡터 검색 결합